In [1]:
# ================================================================
#  LCCN-R21b - PHONETIC GUARD DECOMPOSITION: DEEP AND TRANSFORMER  [GPU]
#  BiLSTM and CNN-BiLSTM (3 seeds, cheap) and XLM-RoBERTa seed 42 on the
#  six single-veto arms (~50 min each, ~5 h). EXTRA_SEEDS adds seeds 43
#  and 44 for the two emotion-only arms (+~3.5 h); leave False unless the
#  seed-42 result is close to the guarded value and needs confirmation.
#  Reference rows (soundex, soundex_guard, indicsoundex, indicsoundex_guard)
#  come from the NB3 and NB4 outputs, byte-identical texts.
#  Add Input: lccn-r21a-guard-arms (output: guard_texts.csv),
#             paper1-nb1-lccn-final (output), paper1-nb2-lccn-ml (output).
#  Accelerator GPU. Internet ON. Save & Run All (Commit).
# ================================================================
import subprocess, sys, os, shutil, glob
def sh(c):
    print("+", c); return subprocess.run(c, shell=True).returncode
sh('pip install -q scikit-learn xgboost scipy')
sh('pip install -q transformers accelerate --upgrade')
for p in ("trainer.py", "/kaggle/working/trainer.py"):
    if os.path.exists(p): os.remove(p)
shutil.rmtree("__pycache__", ignore_errors=True); sys.modules.pop("trainer", None)
tc = [p for p in sorted(glob.glob("/kaggle/input/**/trainer.py", recursive=True)) if "nb2" in p]
tc = tc or sorted(glob.glob("/kaggle/input/**/trainer.py", recursive=True))
assert tc, "attach the paper1-nb2-lccn-ml output"
shutil.copy(tc[0], "/kaggle/working/trainer.py")
import torch
assert torch.cuda.is_available(), "GPU is off - Settings > Accelerator > GPU"
print("gpu:", torch.cuda.get_device_name(0))


+ pip install -q scikit-learn xgboost scipy
+ pip install -q transformers accelerate --upgrade
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 108.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 31.4 MB/s eta 0:00:00
gpu: Tesla T4


In [2]:
# ================================================================
#  CELL 1 - SETUP AND ARM JOIN
# ================================================================
import importlib, trainer
importlib.reload(trainer)
from trainer import *
import pandas as pd, numpy as np
FP = trainer.fingerprint(); assert trainer.TRAINER_VERSION == "1.4" and FP == "5b607248090bd26e", FP
sd = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/splits/train.csv", recursive=True)]
tr, va, te, HAVE = load_splits(sd[0])
gt = glob.glob("/kaggle/input/**/guard_texts.csv", recursive=True); assert gt, "attach lccn-r21a output"
g = pd.read_csv(gt[0], dtype=str, keep_default_na=False); g.columns = [c.strip().lower() for c in g.columns]
idc = next(c for c in g.columns if c in ("id", "comment_id"))
SIX = ["soundex_emotion", "soundex_length", "soundex_negation",
       "indicsoundex_emotion", "indicsoundex_length", "indicsoundex_negation"]
for f in (tr, va, te):
    ids = f["id"].astype(str)
    for a in SIX:
        f[f"text_{a}"] = ids.map(dict(zip(g[idc].astype(str), g[f"text_{a}"]))); assert f[f"text_{a}"].notna().all(), a
print("arms joined:", SIX)


trainer v1.4  sha256:5b607248090bd26e
  BUDGET {'epochs': 10, 'lr': 2e-05, 'batch_size': 32, 'max_len': 96, 'warmup_ratio': 0.1, 'weight_decay': 0.01, 'patience': 3}
  DEEP   {'epochs': 8, 'lr': 0.001, 'batch_size': 64, 'max_len': 64, 'vocab': 40000, 'emb': 200, 'hid': 128, 'dropout': 0.3, 'patience': 2}
test-set id hash: e2d6e1f7d564cace
train  28,000  {'disgust': np.int64(7000), 'happy': np.int64(7000), 'sad': np.int64(7000), 'anger': np.int64(7000)}
val     6,000  {'anger': np.int64(1500), 'happy': np.int64(1500), 'sad': np.int64(1500), 'disgust': np.int64(1500)}
test    6,000  {'disgust': np.int64(1500), 'anger': np.int64(1500), 'sad': np.int64(1500), 'happy': np.int64(1500)}
arms found: ['raw', 'keel', 'kope', 'metaphone', 'terun', 'soundex', 'soundex_guard', 'indicsoundex', 'indicsoundex_guard', 'lccn_base', 'lccn', 'lccn_itrans', 'lccn_indicxlit']
arms joined: ['soundex_emotion', 'soundex_length', 'soundex_negation', 'indicsoundex_emotion', 'indicsoundex_length', 'indicsoundex_n

In [3]:
# ================================================================
#  CELL 2 - DEEP (3 seeds) AND XLM-RoBERTa (seed 42; optional extra seeds)
# ================================================================
EXTRA_SEEDS = False
deep = run_grid(tr, va, te, SIX, ["BiLSTM", "CNN-BiLSTM"], tag="lccn_r21_guard_deep", resume=True)
print(deep.to_string(index=False))
xl = run_grid(tr, va, te, SIX, ["XLM-RoBERTa"], tag="lccn_r21_guard_xlmr", seeds_tf=(42,), resume=True)
print(xl.to_string(index=False))
for a in SIX:
    s = f"/kaggle/working/preds/XLM-RoBERTa__{a}.csv"
    if os.path.exists(s): shutil.copy(s, f"/kaggle/working/preds/XLM-RoBERTa__{a}__seed42.csv")
if EXTRA_SEEDS:
    for seed in (43, 44):
        for a in ("soundex_emotion", "indicsoundex_emotion"):
            p = f"/kaggle/working/preds/XLM-RoBERTa__{a}.csv"
            if os.path.exists(p): os.remove(p)
        r = run_grid(tr, va, te, ["soundex_emotion", "indicsoundex_emotion"], ["XLM-RoBERTa"],
                     tag=f"lccn_r21_guard_xlmr_seed{seed}", seeds_tf=(seed,), resume=False)
        print(r.to_string(index=False))
        for a in ("soundex_emotion", "indicsoundex_emotion"):
            shutil.copy(f"/kaggle/working/preds/XLM-RoBERTa__{a}.csv",
                        f"/kaggle/working/preds/XLM-RoBERTa__{a}__seed{seed}.csv")

REF = {"XLM-RoBERTa": {"soundex": 0.7786, "soundex_guard": 0.7792, "indicsoundex": 0.7773, "indicsoundex_guard": 0.7861}}
r = xl.iloc[0]
print("\nXLM-RoBERTa seed 42: how much of the guarded gain each veto recovers")
for base in ("soundex", "indicsoundex"):
    u, gd = REF["XLM-RoBERTa"][base], REF["XLM-RoBERTa"][base + "_guard"]
    print(f"  {base}: unguarded {u:.4f}  guarded {gd:.4f}  gain {gd-u:+.4f}")
    for v in ("emotion", "length", "negation"):
        val = float(r[f"{base}_{v}"]); share = (val - u) / (gd - u) if gd != u else float("nan")
        print(f"    {v:<9} only: {val:.4f}  ({val-u:+.4f}, {100*share:5.1f}% of the guarded gain)")
print("\nDONE - results_lccn_r21_guard_deep.csv, results_lccn_r21_guard_xlmr.csv, preds/ written.")


resume: 58 prediction files already on disk


=== BiLSTM / soundex_emotion ===
  -> macro-F1 0.7424 +/- 0.0079

=== BiLSTM / soundex_length ===
  -> macro-F1 0.7566 +/- 0.0025

=== BiLSTM / soundex_negation ===
  -> macro-F1 0.7397 +/- 0.0152

=== BiLSTM / indicsoundex_emotion ===
  -> macro-F1 0.7549 +/- 0.0100

=== BiLSTM / indicsoundex_length ===
  -> macro-F1 0.7444 +/- 0.0075

=== BiLSTM / indicsoundex_negation ===
  -> macro-F1 0.7472 +/- 0.0080

=== CNN-BiLSTM / soundex_emotion ===
  -> macro-F1 0.7423 +/- 0.0133

=== CNN-BiLSTM / soundex_length ===
  -> macro-F1 0.7440 +/- 0.0027

=== CNN-BiLSTM / soundex_negation ===
  -> macro-F1 0.7358 +/- 0.0151

=== CNN-BiLSTM / indicsoundex_emotion ===
  -> macro-F1 0.7398 +/- 0.0136

=== CNN-BiLSTM / indicsoundex_length ===
  -> macro-F1 0.7416 +/- 0.0109

=== CNN-BiLSTM / indicsoundex_negation ===
  -> macro-F1 0.7435 +/- 0.0069

wrote /kaggle/working/results_lccn_r21_guard_deep.csv
     model family  soundex_emotion  soundex_emotion_st

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

...

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    epoch  1/10  val macro-F1 = 0.7098
    epoch  2/10  val macro-F1 = 0.7634
    epoch  3/10  val macro-F1 = 0.7787
    epoch  4/10  val macro-F1 = 0.7818
    epoch  5/10  val macro-F1 = 0.7958
    epoch  6/10  val macro-F1 = 0.7867
    epoch  7/10  val macro-F1 = 0.7879
    epoch  8/10  val macro-F1 = 0.7873
    early stop at epoch 8
  -> macro-F1 0.7866

=== XLM-RoBERTa / indicsoundex_negation ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    epoch  1/10  val macro-F1 = 0.7442
    epoch  2/10  val macro-F1 = 0.7640
    epoch  3/10  val macro-F1 = 0.7814
    epoch  4/10  val macro-F1 = 0.7774
    epoch  5/10  val macro-F1 = 0.7881
    epoch  6/10  val macro-F1 = 0.7830
    epoch  7/10  val macro-F1 = 0.7873
    epoch  8/10  val macro-F1 = 0.7797
    early stop at epoch 8
  -> macro-F1 0.7853

wrote /kaggle/working/results_lccn_r21_guard_xlmr.csv
      model      family  soundex_emotion  soundex_length  soundex_negation  indicsoundex_emotion  indicsoundex_length  indicsoundex_negation
XLM-RoBERTa transformer           0.7857          0.7847             0.785                0.7845               0.7866                 0.7853

XLM-RoBERTa seed 42: how much of the guarded gain each veto recovers
  soundex: unguarded 0.7786  guarded 0.7792  gain +0.0006
    emotion   only: 0.7857  (+0.0071, 1183.3% of the guarded gain)
    length    only: 0.7847  (+0.0061, 1016.7% of the guarded gain)
    negation  only: 0.7850  (+0.0064, 1066